# 00 · Revisión del entorno

Este cuaderno lo ejecuto una vez en cada máquina donde trabajo el proyecto: mi Mac con
VS Code y, más adelante, Colab. No procesa datos. Responde tres preguntas: qué versiones
tengo instaladas, si PyTorch ve una GPU, y cuánta memoria y disco quedan. La salida la
copio en la bitácora, para que quede registro de en qué condiciones corrió cada etapa.

Por qué me importa: el plan reparte el trabajo en dos lugares. Todo lo que sea leer
DICOM, calcular SUV, remuestrear y evaluar lo hago en el Mac. Los entrenamientos con el
presupuesto completo (25 000 iteraciones por modelo) van a Colab, salvo que el Mac tenga
memoria de sobra y las pruebas cortas muestren que rinde. Esta revisión es la que decide.

In [ ]:
import sys, platform, importlib, shutil, os

print("Python", sys.version.split()[0], "en", platform.platform())
print("Procesador:", platform.machine())

paquetes = ["numpy", "scipy", "pandas", "pydicom", "SimpleITK", "nibabel",
            "skimage", "matplotlib", "yaml", "pytest", "torch", "monai", "tcia_utils"]
for nombre in paquetes:
    try:
        mod = importlib.import_module(nombre)
        print(f"  {nombre:12s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {nombre:12s} NO instalado")

## ¿Hay GPU?

En Colab espero ver `cuda`. En mi Mac con chip M la respuesta es `mps`, que es el nombre
que PyTorch le da al chip gráfico de Apple. Si dijera `cpu`, el entrenamiento igual
funciona, solo que mucho más lento.

In [ ]:
import sys
sys.path.insert(0, "../src")
from petct.device import describe_device
print("Dispositivo que usaría PyTorch:", describe_device())

## Memoria y disco

La regla práctica que uso: los 250 estudios en NIfTI a resolución nativa ocupan unos
25 GB; a 3 mm, menos de 8 GB. Un parche 3D de 96³ con dos canales y lote de 2 pesa poco,
pero la U-Net guarda activaciones intermedias, y ahí se van varios GB. Con 16 GB de
memoria unificada alcanza para probar; para el presupuesto completo prefiero Colab o un
Mac con 32 GB o más.

In [ ]:
import shutil, os
total, usado, libre = shutil.disk_usage(os.getcwd())
print(f"Disco libre en esta carpeta: {libre / 2**30:.1f} GB de {total / 2**30:.1f} GB")
try:
    import psutil
    print(f"Memoria RAM total: {psutil.virtual_memory().total / 2**30:.1f} GB")
except ImportError:
    print("psutil no instalado (en macOS la RAM también se obtiene con: sysctl hw.memsize)")

## Lo que anoto en la bitácora

Fecha, máquina, versión de Python, dispositivo (`cuda`, `mps` o `cpu`), RAM y disco
libre. Con eso, quien lea el informe sabe en qué condiciones corrió cada etapa, y yo sé
si el Paso 3 lo hago aquí o en Colab.